# MLP para classificação de diabetes

Notebook autocontido para treino, avaliação, inferência e validações CPU/CUDA.

## Ambiente

Instale primeiro `pip install -r requirements.txt` e então **uma** das opções:

- RTX 3050/CUDA 13.0: `pip install -r requirements-cuda.txt`
- Sem GPU: `pip install -r requirements-cpu.txt`

Execute as células em ordem. A configuração padrão usa CUDA quando disponível.

## Configuração e reprodutibilidade

In [ ]:
"""Central configuration for training and artifact generation."""
import os
from dataclasses import asdict, dataclass
from pathlib import Path


PROJECT_ROOT = Path.cwd()


@dataclass(frozen=True)
class Config:
    seed: int = 42
    data_path: Path = PROJECT_ROOT / "dataset" / "diabetes_prediction_dataset.csv"
    artifacts_dir: Path = PROJECT_ROOT / "artifacts"
    checkpoint_path: Path = PROJECT_ROOT / "artifacts" / "best_model.pt"
    preprocessor_path: Path = PROJECT_ROOT / "artifacts" / "preprocessor.joblib"
    train_size: float = 0.70
    validation_size: float = 0.15
    test_size: float = 0.15
    batch_size: int = 512
    epochs: int = 30
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    hidden_dims: tuple[int, ...] = (128, 64, 32)
    dropout: float = 0.20
    num_classes: int = 2
    early_stopping_patience: int = 6
    monitor: str = "val_loss"
    log_interval: int = 1
    device_override: str | None = None
    max_rows: int | None = None
    use_class_weights: bool = False

    def validate(self) -> None:
        if abs(self.train_size + self.validation_size + self.test_size - 1.0) > 1e-9:
            raise ValueError("As frações de treino, validação e teste devem somar 1.0.")
        if min(self.train_size, self.validation_size, self.test_size) <= 0:
            raise ValueError("Todas as frações devem ser positivas.")
        if self.num_classes != 2:
            raise ValueError("Este projeto espera exatamente duas classes.")

    def as_dict(self) -> dict[str, object]:
        payload = asdict(self)
        return {key: str(value) if isinstance(value, Path) else value for key, value in payload.items()}


def get_config() -> Config:
    """Build config, allowing explicit environment overrides for smoke tests."""
    def optional_int(name: str) -> int | None:
        value = os.getenv(name)
        return int(value) if value else None

    artifacts_override = os.getenv("MLP_ARTIFACTS_DIR")
    artifacts_dir = Path(artifacts_override) if artifacts_override else Config.artifacts_dir
    config = Config(
        artifacts_dir=artifacts_dir,
        checkpoint_path=artifacts_dir / "best_model.pt",
        preprocessor_path=artifacts_dir / "preprocessor.joblib",
        epochs=int(os.getenv("MLP_EPOCHS", Config.epochs)),
        batch_size=int(os.getenv("MLP_BATCH_SIZE", Config.batch_size)),
        device_override=os.getenv("MLP_DEVICE") or None,
        max_rows=optional_int("MLP_MAX_ROWS"),
        use_class_weights=os.getenv("MLP_CLASS_WEIGHTS", "0").lower() in {"1", "true", "yes"},
    )
    config.validate()
    return config


"""Runtime utilities for deterministic execution and metadata."""
import platform
import random
import sys

import numpy as np
import torch


def set_seed(seed: int) -> None:
    """Seed every RNG used by the pipeline before splitting or model creation."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    # Determinism can reduce CUDA throughput, but makes runs comparable.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except AttributeError:  # older supported PyTorch releases
        pass


def get_device(override: str | None = None) -> torch.device:
    if override:
        requested = torch.device(override)
        if requested.type == "cuda" and not torch.cuda.is_available():
            raise RuntimeError("MLP_DEVICE=cuda foi solicitado, mas CUDA não está disponível.")
        return requested
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def runtime_metadata(device: torch.device) -> dict[str, object]:
    metadata: dict[str, object] = {
        "python": sys.version,
        "platform": platform.platform(),
        "torch": torch.__version__,
        "torch_cuda_version": torch.version.cuda,
        "device": str(device),
        "cuda_available": torch.cuda.is_available(),
    }
    if device.type == "cuda":
        metadata.update({
            "gpu_name": torch.cuda.get_device_name(device),
            "cuda_device_count": torch.cuda.device_count(),
        })
    return metadata


## Dados e modelo

O pré-processador é ajustado exclusivamente no treino; validação e teste usam apenas `transform`.

In [ ]:
"""Dataset validation, leakage-free preprocessing and PyTorch loaders."""
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset


CATEGORICAL_COLUMNS = ["gender", "smoking_history"]
NUMERIC_COLUMNS = ["age", "hypertension", "heart_disease", "bmi", "HbA1c_level", "blood_glucose_level"]
TARGET_COLUMN = "diabetes"
REQUIRED_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS + [TARGET_COLUMN]


@dataclass
class Splits:
    x_train: pd.DataFrame
    x_val: pd.DataFrame
    x_test: pd.DataFrame
    y_train: pd.Series
    y_val: pd.Series
    y_test: pd.Series


@dataclass
class PreparedData:
    loaders: dict[str, DataLoader]
    preprocessor: ColumnTransformer
    in_dim: int
    splits: Splits
    split_summary: dict[str, dict[str, float | int]]


def load_dataset(path: Path, max_rows: int | None = None) -> pd.DataFrame:
    """Read and validate the expected CSV schema and binary target."""
    if not path.is_file():
        raise FileNotFoundError(f"Dataset não encontrado: {path}")
    frame = pd.read_csv(path, nrows=max_rows)
    missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(f"CSV incompatível; colunas ausentes: {missing}. Encontradas: {list(frame.columns)}")
    if frame.empty:
        raise ValueError("O dataset está vazio.")
    nulls = frame[REQUIRED_COLUMNS].isna().sum()
    if int(nulls.sum()) > 0:
        details = {key: int(value) for key, value in nulls.items() if value}
        raise ValueError(f"O dataset contém valores ausentes. Política configurada: rejeitar. Detalhes: {details}")
    target = frame[TARGET_COLUMN]
    if not set(target.unique()).issubset({0, 1}):
        raise ValueError(f"O alvo deve conter somente 0 e 1; encontrados: {sorted(target.unique().tolist())}")
    if target.nunique() != 2:
        raise ValueError("A divisão estratificada requer as duas classes no alvo.")
    return frame[REQUIRED_COLUMNS].copy()


def dataset_diagnostics(frame: pd.DataFrame) -> dict[str, object]:
    return {
        "rows": int(len(frame)),
        "dtypes": {column: str(dtype) for column, dtype in frame.dtypes.items()},
        "nulls": {column: int(count) for column, count in frame.isna().sum().items()},
        "target_distribution": {str(label): int(count) for label, count in frame[TARGET_COLUMN].value_counts().sort_index().items()},
    }


def split_data(frame: pd.DataFrame, config: Config) -> Splits:
    x = frame.drop(columns=TARGET_COLUMN)
    y = frame[TARGET_COLUMN].astype("int64")
    x_remaining, x_test, y_remaining, y_test = train_test_split(
        x, y, test_size=config.test_size, random_state=config.seed, stratify=y
    )
    validation_relative_size = config.validation_size / (config.train_size + config.validation_size)
    x_train, x_val, y_train, y_val = train_test_split(
        x_remaining, y_remaining, test_size=validation_relative_size,
        random_state=config.seed, stratify=y_remaining,
    )
    return Splits(x_train, x_val, x_test, y_train, y_val, y_test)


def build_preprocessor() -> ColumnTransformer:
    # Dense output is deliberate: the MLP consumes dense float32 tensors.
    categorical = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    numeric = Pipeline([("scaler", StandardScaler())])
    return ColumnTransformer(
        transformers=[("categorical", categorical, CATEGORICAL_COLUMNS), ("numeric", numeric, NUMERIC_COLUMNS)],
        remainder="drop",
        sparse_threshold=0.0,
    )


def _as_float32(matrix: object) -> np.ndarray:
    return np.asarray(matrix, dtype=np.float32)


def _summary(splits: Splits) -> dict[str, dict[str, float | int]]:
    result: dict[str, dict[str, float | int]] = {}
    for name, target in (("train", splits.y_train), ("validation", splits.y_val), ("test", splits.y_test)):
        result[name] = {"rows": int(len(target)), "positive_rate": float(target.mean()), "positive_count": int(target.sum())}
    return result


def prepare_data(frame: pd.DataFrame, config: Config, device: torch.device) -> PreparedData:
    splits = split_data(frame, config)
    preprocessor = build_preprocessor()
    # This is the sole fit in the data path: validation/test use only transform.
    transformed = {
        "train": _as_float32(preprocessor.fit_transform(splits.x_train)),
        "validation": _as_float32(preprocessor.transform(splits.x_val)),
        "test": _as_float32(preprocessor.transform(splits.x_test)),
    }
    widths = {name: values.shape[1] for name, values in transformed.items()}
    if len(set(widths.values())) != 1:
        raise RuntimeError(f"Dimensões transformadas incompatíveis: {widths}")
    labels = {"train": splits.y_train, "validation": splits.y_val, "test": splits.y_test}
    generator = torch.Generator().manual_seed(config.seed)
    loaders: dict[str, DataLoader] = {}
    for name in ("train", "validation", "test"):
        target_values = labels[name].to_numpy(dtype=np.int64, copy=True)
        dataset = TensorDataset(torch.from_numpy(transformed[name]), torch.from_numpy(target_values))
        loaders[name] = DataLoader(
            dataset, batch_size=config.batch_size, shuffle=name == "train", generator=generator if name == "train" else None,
            pin_memory=device.type == "cuda", num_workers=0,
        )
    return PreparedData(loaders, preprocessor, widths["train"], splits, _summary(splits))


"""Configurable multi-layer perceptron."""
import torch
from torch import nn


class MLP(nn.Module):
    def __init__(self, in_dim: int, hidden_dims: tuple[int, ...] | list[int], num_classes: int, dropout: float) -> None:
        super().__init__()
        layers: list[nn.Module] = []
        previous = in_dim
        for hidden in hidden_dims:
            layers.extend([nn.Linear(previous, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout)])
            previous = hidden
        layers.append(nn.Linear(previous, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


## Alternativa didática: subclasse de `Dataset`

O pipeline atual usa `TensorDataset` em `prepare_data`, pois atributos e rótulos já foram convertidos para tensores. Uma subclasse de `Dataset` só é necessária quando cada amostra exige comportamento próprio, como transformações sob demanda, carregamento preguiçoso ou metadados adicionais.

Caso essa abstração seja desejada, faça três alterações **na célula Dados e modelo**, sem mudar as rotinas de treino, validação ou os `DataLoader`s:

1. Troque o import por `from torch.utils.data import DataLoader, Dataset`.
2. Logo após `PreparedData` e antes de `prepare_data`, adicione:

```python
class DiabetesDataset(Dataset):
    def __init__(self, features: np.ndarray, labels: np.ndarray) -> None:
        self.features = torch.as_tensor(features, dtype=torch.float32)
        self.labels = torch.as_tensor(labels, dtype=torch.long)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.features[index], self.labels[index]
```

3. No laço de `prepare_data`, substitua a linha que cria `TensorDataset` por:

```python
dataset = DiabetesDataset(transformed[name], target_values)
```

O restante da criação de cada `DataLoader` permanece igual. Para a representação densa e inteiramente carregada deste projeto, a troca não altera desempenho ou resultado; por isso `TensorDataset` continua sendo a implementação ativa e mais simples.

## Treino, avaliação e inferência

In [ ]:
"""Training, validation, checkpointing and early stopping."""
from pathlib import Path
from typing import Any

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score
from torch import nn
from torch.utils.data import DataLoader



def _move(batch: tuple[torch.Tensor, torch.Tensor], device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    non_blocking = device.type == "cuda"
    return batch[0].to(device, non_blocking=non_blocking), batch[1].to(device, non_blocking=non_blocking)


def train_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, optimizer: torch.optim.Optimizer, device: torch.device) -> float:
    model.train()
    loss_sum = 0.0
    examples = 0
    for batch in loader:
        features, labels = _move(batch, device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(features), labels)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * labels.size(0)
        examples += labels.size(0)
    return loss_sum / examples


def validate_epoch(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> dict[str, float]:
    model.eval()
    loss_sum = 0.0
    examples = 0
    predicted: list[int] = []
    actual: list[int] = []
    with torch.no_grad():
        for batch in loader:
            features, labels = _move(batch, device)
            logits = model(features)
            loss_sum += criterion(logits, labels).item() * labels.size(0)
            examples += labels.size(0)
            predicted.extend(logits.argmax(dim=1).cpu().tolist())
            actual.extend(labels.cpu().tolist())
    return {
        "loss": loss_sum / examples,
        "accuracy": float(accuracy_score(actual, predicted)),
        "f1": float(f1_score(actual, predicted, pos_label=1, zero_division=0)),
    }


def train(model: nn.Module, loaders: dict[str, DataLoader], config: Config, device: torch.device, class_weights: torch.Tensor | None = None) -> dict[str, list[float]]:
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device) if class_weights is not None else None)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    history: dict[str, list[float]] = {"train_loss": [], "val_loss": [], "val_acc": [], "val_f1": []}
    best_loss = float("inf")
    epochs_without_improvement = 0
    config.checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    for epoch in range(1, config.epochs + 1):
        train_loss = train_epoch(model, loaders["train"], criterion, optimizer, device)
        validation = validate_epoch(model, loaders["validation"], criterion, device)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(validation["loss"])
        history["val_acc"].append(validation["accuracy"])
        history["val_f1"].append(validation["f1"])
        if epoch % config.log_interval == 0:
            print(f"epoch={epoch:03d} train_loss={train_loss:.5f} val_loss={validation['loss']:.5f} val_acc={validation['accuracy']:.4f} val_f1={validation['f1']:.4f}")
        if validation["loss"] < best_loss:
            best_loss = validation["loss"]
            epochs_without_improvement = 0
            torch.save({
                "state_dict": model.state_dict(), "epoch": epoch, "best_val_loss": best_loss,
                "history": history, "in_dim": model.net[0].in_features, "hidden_dims": list(config.hidden_dims),
                "num_classes": config.num_classes, "dropout": config.dropout, "config": config.as_dict(),
            }, config.checkpoint_path)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= config.early_stopping_patience:
                print(f"Early stopping na época {epoch}; paciência={config.early_stopping_patience}.")
                break
    checkpoint: dict[str, Any] = torch.load(config.checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint["state_dict"])
    return history


"""Final evaluation and visualization helpers."""
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from torch import nn
from torch.utils.data import DataLoader


def evaluate(model: nn.Module, loader: DataLoader, criterion: nn.Module, device: torch.device) -> tuple[dict[str, object], np.ndarray, np.ndarray]:
    model.eval()
    loss_sum, examples = 0.0, 0
    actual: list[int] = []
    predicted: list[int] = []
    with torch.no_grad():
        for features, labels in loader:
            features, labels = features.to(device, non_blocking=device.type == "cuda"), labels.to(device, non_blocking=device.type == "cuda")
            logits = model(features)
            loss_sum += criterion(logits, labels).item() * labels.size(0)
            examples += labels.size(0)
            actual.extend(labels.cpu().tolist())
            predicted.extend(logits.argmax(dim=1).cpu().tolist())
    y_true, y_pred = np.asarray(actual), np.asarray(predicted)
    metrics: dict[str, object] = {
        "loss": loss_sum / examples,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_positive": float(precision_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "recall_positive": float(recall_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "f1_positive": float(f1_score(y_true, y_pred, pos_label=1, zero_division=0)),
        "classification_report": classification_report(y_true, y_pred, output_dict=True, zero_division=0),
    }
    return metrics, y_true, y_pred


def save_metrics(metrics: dict[str, object], y_true: np.ndarray, y_pred: np.ndarray, artifacts_dir: Path) -> None:
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    (artifacts_dir / "test_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    report = classification_report(y_true, y_pred, zero_division=0)
    (artifacts_dir / "classification_report.txt").write_text(report, encoding="utf-8")
    matrix = confusion_matrix(y_true, y_pred, labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", xticklabels=[0, 1], yticklabels=[0, 1], ax=ax)
    ax.set(xlabel="Predito", ylabel="Real", title="Matriz de confusão — teste")
    fig.tight_layout()
    fig.savefig(artifacts_dir / "confusion_matrix.png", dpi=160)
    plt.close(fig)


def save_learning_curves(history: dict[str, list[float]], artifacts_dir: Path) -> None:
    epochs = range(1, len(history["train_loss"]) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(epochs, history["train_loss"], label="Treino")
    axes[0].plot(epochs, history["val_loss"], label="Validação")
    axes[0].set(title="Perda", xlabel="Época", ylabel="Cross-entropy")
    axes[1].plot(epochs, history["val_acc"], label="Acurácia")
    axes[1].plot(epochs, history["val_f1"], label="F1 positiva")
    axes[1].set(title="Métricas de validação", xlabel="Época", ylabel="Valor")
    for axis in axes:
        axis.grid(True, alpha=.3)
        axis.legend()
    fig.tight_layout()
    fig.savefig(artifacts_dir / "learning_curves.png", dpi=160)
    plt.close(fig)


"""Artifact-only inference: no training data or retraining is required."""
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch



FEATURE_COLUMNS = CATEGORICAL_COLUMNS + NUMERIC_COLUMNS


def load_predictor(artifacts_dir: Path, device: torch.device) -> tuple[MLP, object]:
    checkpoint = torch.load(artifacts_dir / "best_model.pt", map_location=device, weights_only=False)
    model = MLP(checkpoint["in_dim"], checkpoint["hidden_dims"], checkpoint["num_classes"], checkpoint["dropout"]).to(device)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    return model, joblib.load(artifacts_dir / "preprocessor.joblib")


def predict(frame: pd.DataFrame, artifacts_dir: Path, device: torch.device) -> np.ndarray:
    missing = sorted(set(FEATURE_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(f"Entradas sem colunas exigidas: {missing}")
    model, preprocessor = load_predictor(artifacts_dir, device)
    features = np.asarray(preprocessor.transform(frame[FEATURE_COLUMNS]), dtype=np.float32)
    with torch.no_grad():
        return model(torch.from_numpy(features).to(device)).argmax(dim=1).cpu().numpy()


## Execução normal

`run_experiment()` grava checkpoint, pré-processador, métricas, relatório e gráficos. Para um teste curto, informe `max_rows=3000, epochs=3`; para CUDA use `device_override='cuda'`.

In [ ]:
def run_experiment(
    *,
    device_override: str | None = None,
    max_rows: int | None = None,
    epochs: int | None = None,
    artifacts_dir: Path | None = None,
    use_class_weights: bool = False,
) -> dict[str, object]:
    """Execute o fluxo completo sem depender de qualquer arquivo .py externo."""
    base = Config()
    target_dir = artifacts_dir or base.artifacts_dir
    config = Config(
        artifacts_dir=target_dir, checkpoint_path=target_dir / 'best_model.pt',
        preprocessor_path=target_dir / 'preprocessor.joblib',
        device_override=device_override, max_rows=max_rows,
        epochs=epochs if epochs is not None else base.epochs,
        use_class_weights=use_class_weights,
    )
    config.validate()
    set_seed(config.seed)
    device = get_device(config.device_override)
    metadata = runtime_metadata(device)
    print(f"Dispositivo selecionado: {metadata['device']}")
    if device.type == 'cuda':
        print(f"GPU: {metadata['gpu_name']} | CUDA PyTorch: {metadata['torch_cuda_version']}")
    else:
        print('CUDA indisponível ou não solicitado; usando CPU.')
    target_dir.mkdir(parents=True, exist_ok=True)
    frame = load_dataset(config.data_path, config.max_rows)
    prepared = prepare_data(frame, config, device)
    print(f"Partições: {prepared.split_summary}; in_dim={prepared.in_dim}")
    model = MLP(prepared.in_dim, config.hidden_dims, config.num_classes, config.dropout).to(device)
    xb, yb = next(iter(prepared.loaders['train']))
    with torch.no_grad():
        logits = model(xb.to(device, non_blocking=device.type == 'cuda'))
    if logits.shape != (xb.shape[0], config.num_classes) or not torch.isfinite(logits).all():
        raise RuntimeError('Validação do forward da MLP falhou.')
    if device.type == 'cuda' and not prepared.loaders['train'].pin_memory:
        raise RuntimeError('pin_memory deveria estar ativo em CUDA.')
    class_weights = None
    if config.use_class_weights:
        counts = np.bincount(prepared.splits.y_train.to_numpy(), minlength=config.num_classes)
        class_weights = torch.tensor(len(prepared.splits.y_train) / (config.num_classes * counts), dtype=torch.float32)
    history = train(model, prepared.loaders, config, device, class_weights)
    criterion = torch.nn.CrossEntropyLoss(weight=class_weights.to(device) if class_weights is not None else None)
    metrics, y_true, y_pred = evaluate(model, prepared.loaders['test'], criterion, device)
    joblib.dump(prepared.preprocessor, config.preprocessor_path)
    save_metrics(metrics, y_true, y_pred, target_dir)
    save_learning_curves(history, target_dir)
    sample_prediction = predict(prepared.splits.x_test.head(3), target_dir, device).tolist()
    metadata.update({
        'config': config.as_dict(), 'diagnostics': dataset_diagnostics(frame),
        'splits': prepared.split_summary, 'history': history, 'test_metrics': metrics,
        'inference_smoke_predictions': sample_prediction,
        'class_weighting_enabled': config.use_class_weights,
        'imbalance_conclusion': 'Acurácia não é suficiente; recall e F1 da classe positiva devem ser avaliados junto dela.',
    })
    (target_dir / 'metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
    (target_dir / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
    (target_dir / 'imbalance_report.txt').write_text(
        f"Taxa positiva treino: {prepared.split_summary['train']['positive_rate']:.5f}\\n"
        f"Taxa positiva teste: {prepared.split_summary['test']['positive_rate']:.5f}\\n"
        f"Acurácia: {metrics['accuracy']:.5f}\\nPrecision positiva: {metrics['precision_positive']:.5f}\\n"
        f"Recall positivo: {metrics['recall_positive']:.5f}\\nF1 positivo: {metrics['f1_positive']:.5f}\\n"
        'Conclusão: em classes desbalanceadas, acurácia isolada não é critério suficiente.\\n', encoding='utf-8'
    )
    print(f"Métricas de teste: {json.dumps(metrics, ensure_ascii=False)}")
    print(f"Inferência recarregada (3 amostras): {sample_prediction}")
    return metadata

# Execução completa padrão (30 épocas, CUDA automático):
# result = run_experiment()

# Execução curta para inspeção inicial:
# result = run_experiment(max_rows=3000, epochs=3)


## Validações rápidas

Execute as células abaixo depois da definição de `run_experiment`.

In [ ]:
def verify_reproducibility() -> dict[str, object]:
    """Duas execuções CPU idênticas, isoladas em diretórios de artefatos."""
    root = Path('artifacts')
    first = run_experiment(device_override='cpu', max_rows=3000, epochs=3, artifacts_dir=root / 'repro_run_1')
    second = run_experiment(device_override='cpu', max_rows=3000, epochs=3, artifacts_dir=root / 'repro_run_2')
    for key in ('splits', 'history', 'test_metrics', 'inference_smoke_predictions'):
        if first[key] != second[key]:
            raise AssertionError(f'Execuções com mesma semente divergem em: {key}')
    summary = {
        'device': 'cpu', 'fast_mode': True,
        'runtime': {key: first[key] for key in ('python', 'torch', 'torch_cuda_version', 'cuda_available')},
        'split_sizes': {name: value['rows'] for name, value in first['splits'].items()},
        'test_metrics': first['test_metrics'], 'result': 'identical deterministic CPU runs',
    }
    (root / 'reproducibility_check.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return summary

def verify_cuda() -> dict[str, object]:
    """Teste curto real em CUDA; falha claramente caso a GPU não esteja disponível."""
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA não está disponível neste ambiente.')
    return run_experiment(device_override='cuda', max_rows=3000, epochs=3, artifacts_dir=Path('artifacts') / 'cuda_validation')

# cpu_check = verify_reproducibility()
# cuda_check = verify_cuda()
